## Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics
## Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
- Answer relevance
- Answer accuracy
- Retrieval quality
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

langsmith_key = os.getenv("LANGSMITH_API_KEY")

if langsmith_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_key

os.environ["LANGSMITH_TRACING"] = "true"

In [ ]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

## Define Metrics (LLM As A Judge)

In [ ]:
import requests

def ask_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

In [ ]:
def correctness(inputs, outputs, reference_outputs):
    prompt = f"""
You are a strict grader.

Compare the two answers:

Correct Answer: {reference_outputs['answer']}
Student Answer: {outputs['response']}

Rules:
- If both mean the same → CORRECT
- Otherwise → INCORRECT
- Output ONLY one word

Answer:
"""

    response = ask_llm(prompt)

    print("RAW RESPONSE:", response)

    response = response.strip().upper()

    if "INCORRECT" in response:
        return False
    elif "CORRECT" in response:
        return True
    else:
        return False

In [ ]:
print(correctness(
    {"question": "What is 2+2?"},
    {"response": "4"},
    {"answer": "4"}
))

In [ ]:
import requests

default_instructions = "Respond to the user's question in one short sentence."

def my_app(question: str, model: str = "llama3", instructions: str = default_instructions) -> str:
    prompt = f"""
{instructions}

Question: {question}
Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    return response.json()["response"].strip()

In [ ]:
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(inputs["question"])
    }

In [ ]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness],   # ✅ only this
    experiment_prefix="llama3-chatbot"
)

In [ ]:
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(inputs["question"], model="llama3")
    }

In [ ]:
print(ls_target({"question": "What is 2+2?"}))

In [ ]:
def concision(inputs, outputs) -> bool:
    return len(outputs["response"].split()) <= 15

In [ ]:
experiment_results = client.evaluate(
    ls_target,   # Ollama-based app
    data=dataset_name,
    evaluators=[correctness, concision],  # ensure both exist
    experiment_prefix="llama3-chatbot"   # ✅ updated name
)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

# URLs
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split text
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

# ✅ FREE embedding model
# from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # ✅ FORCE CPU
)

# Create vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model,
)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [ ]:
retriever.invoke("what is agents")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3",
    temperature=0
)

llm

In [ ]:
from langsmith import traceable
from langchain_core.messages import HumanMessage, SystemMessage

@traceable()
def rag_bot(question: str) -> dict:
    # 🔍 Retrieve relevant docs
    docs = retriever.invoke(question)
    
    # Limit context (important)
    docs = docs[:4]
    
    docs_string = "\n\n".join(doc.page_content for doc in docs)

    # 🧠 Strong prompt
    instructions = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "I don't know."

Answer in maximum 3 concise sentences.

Context:
{docs_string}
"""

    # ✅ Proper message format
    ai_msg = llm.invoke([
        SystemMessage(content=instructions),
        HumanMessage(content=question),
    ])

    return {
        "answer": ai_msg.content,
        "documents": docs
    }

In [ ]:
rag_bot("What is agents")

In [ ]:
from langsmith import Client

client = Client()

dataset_name = "RAG Test Evaluation"

# ✅ Safe dataset creation
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
except:
    dataset = client.create_dataset(dataset_name=dataset_name)

# Examples
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection?"},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions like Wikipedia search and then reasoning over the results."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "Majority label bias, recency bias, and common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Token manipulation, gradient-based attack, jailbreak prompting, human red-teaming, and model red-teaming."},
    }
]

# ✅ Add examples
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# ✅ Ollama LLM
grader_llm = ChatOllama(
    model="llama3",
    temperature=0
)

correctness_instructions = """You are a strict teacher grading a quiz.

You will be given:
- QUESTION
- GROUND TRUTH ANSWER
- STUDENT ANSWER

Rules:
- Check factual correctness only
- Extra correct info is allowed
- Conflicting info → incorrect

Return ONLY in this format:

EXPLANATION: <your reasoning>
FINAL: CORRECT or INCORRECT
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""

    response = grader_llm.invoke([
        SystemMessage(content=correctness_instructions),
        HumanMessage(content=answers)
    ])

    text = response.content.strip().upper()

    print("GRADE RAW:", text)  # debug

    if "INCORRECT" in text:
        return False
    elif "CORRECT" in text:
        return True
    else:
        return False

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# ✅ Ollama LLM
relevance_llm = ChatOllama(
    model="llama3",
    temperature=0
)

relevance_instructions = """You are a strict teacher grading a quiz.

You will be given:
- QUESTION
- STUDENT ANSWER

Rules:
- Answer must be relevant to the question
- Answer must help address the question
- Short and useful answers are better

Return ONLY in this format:

EXPLANATION: <your reasoning>
FINAL: RELEVANT or NOT_RELEVANT
"""

def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
"""

    response = relevance_llm.invoke([
        SystemMessage(content=relevance_instructions),
        HumanMessage(content=answer)
    ])

    text = response.content.strip().upper()

    print("RELEVANCE RAW:", text)  # debug

    if "NOT_RELEVANT" in text:
        return False
    elif "RELEVANT" in text:
        return True
    else:
        return False

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# ✅ Ollama LLM
grounded_llm = ChatOllama(
    model="llama3",
    temperature=0
)

grounded_instructions = """You are a strict teacher grading a quiz.

You will be given:
- FACTS (retrieved documents)
- STUDENT ANSWER

Rules:
- Answer must be fully supported by the facts
- No hallucination (no extra unsupported info)
- If answer contains info not in facts → NOT_GROUNDED

Return ONLY in this format:

EXPLANATION: <your reasoning>
FINAL: GROUNDED or NOT_GROUNDED
"""

def groundedness(inputs: dict, outputs: dict) -> bool:
    # 🔥 limit context (important)
    docs = outputs["documents"][:4]

    doc_string = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
FACTS:
{doc_string}

STUDENT ANSWER:
{outputs['answer']}
"""

    response = grounded_llm.invoke([
        SystemMessage(content=grounded_instructions),
        HumanMessage(content=prompt)
    ])

    text = response.content.strip().upper()

    print("GROUNDED RAW:", text)  # debug

    if "NOT_GROUNDED" in text:
        return False
    elif "GROUNDED" in text:
        return True
    else:
        return False

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# ✅ Ollama LLM
retrieval_relevance_llm = ChatOllama(
    model="llama3",
    temperature=0
)

retrieval_relevance_instructions = """You are a strict teacher grading a quiz.

You will be given:
- QUESTION
- FACTS (retrieved documents)

Rules:
- If ANY part of the facts is relevant → RELEVANT
- If facts are completely unrelated → NOT_RELEVANT
- Partial relevance is OK

Return ONLY in this format:

EXPLANATION: <your reasoning>
FINAL: RELEVANT or NOT_RELEVANT
"""

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    # 🔥 limit docs (important)
    docs = outputs["documents"][:4]

    doc_string = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
QUESTION:
{inputs['question']}

FACTS:
{doc_string}
"""

    response = retrieval_relevance_llm.invoke([
        SystemMessage(content=retrieval_relevance_instructions),
        HumanMessage(content=prompt)
    ])

    text = response.content.strip().upper()

    print("RETRIEVAL RAW:", text)  # debug

    if "NOT_RELEVANT" in text:
        return False
    elif "RELEVANT" in text:
        return True
    else:
        return False

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        retrieval_relevance
    ],
    experiment_prefix="llama3-rag-eval",   # ✅ updated
    metadata={
        "model": "llama3",
        "embedding": "all-MiniLM-L6-v2",
        "pipeline": "RAG + LangChain + Ollama"
    },
)

# Convert to dataframe
df = experiment_results.to_pandas()
df.head()